# Training and testing set construction

#### Basic initial processing for coherence between datasets

In [1]:
import pandas as pd
import numpy as np

def merge_gene_column(df_primary, df_secondary):
    df_primary['ID'] = df_primary['ID'].astype(str)
    df_secondary['ID'] = df_secondary['ID'].astype(str)

    if "ensg" in df_primary.columns:
        df_primary['ID'] = df_primary['ID'].astype(str)
        df_secondary['ID'] = df_secondary['ID'].astype(str)

    df_primary = df_primary.drop(columns=['Gene', 'gene'], errors='ignore')

    # Drop variants with splice altering mutations (because we are only interested in true missense variants)
    valid_ids = df_secondary[df_secondary['spliceai'].isna()]['ID']
    df_primary = df_primary[df_primary['ID'].isin(valid_ids)]
    
    df_primary = df_primary[df_primary['ID'].isin(df_secondary['ID'])]
    
    df_primary = df_primary.merge(df_secondary[['ID', 'Gene', "ensg"]], on=(['ID', 'ensg'] if 'ensg' in df_primary.columns else 'ID'), how='left')
    
    return df_primary

curation = pd.read_csv("../data/datasets/functional/curation.txt", sep='\t', low_memory=False)
text_mining = pd.read_csv("../data/datasets/functional/text_mining.txt", sep='\t', low_memory=False)
mave = pd.read_csv("../data/datasets/functional/mave.txt", sep='\t', low_memory=False)
proxy = pd.read_parquet("../data/datasets/population/population_missense_variant_set.parquet")
clinvar = pd.read_csv("../data/datasets/clinvar/clinvar_missense_2025_2026.txt", sep='\t', low_memory=False) # ClinVar data from 2025-01-21 to 2025-11-30 (new data used for testing)
clinvar_jan2025 = pd.read_csv("../data/datasets/clinvar/clinvar_missense_jan2025.txt", sep='\t', low_memory=False) # ClinVar data at 2025-01-21 (used for training ClinVEP)
variant_set = pd.read_parquet('../data/intermediate/feature_matrix.parquet', columns=["ID", "gene", "spliceai", "ensg"]).rename(columns={"gene": "Gene"})

curation = merge_gene_column(curation, variant_set)
text_mining = merge_gene_column(text_mining, variant_set)
mave = merge_gene_column(mave, variant_set)
clinvar = merge_gene_column(clinvar, variant_set)
clinvar_jan2025 = merge_gene_column(clinvar_jan2025, variant_set)
proxy = merge_gene_column(proxy, variant_set)

variant_set = variant_set[["ID", "ensg"]]

## Construct a balanced functional training dataset

#### Remove external evaluation gene families (all homologs) from training datasets in order to fully avoid type 2 circularity

In [2]:
gene_families = pd.read_csv("../resources/gene_families/gene_families.txt", sep="\t")
training_genes = gene_families[gene_families["Training_set"] == 1]["ensg"].tolist()

def filter_to_training_genes(df, training_genes):
    return df[df["ensg"].isin(training_genes)]

text_mining_training = filter_to_training_genes(text_mining, training_genes)
mave_training = filter_to_training_genes(mave, training_genes)
curation_training = filter_to_training_genes(curation, training_genes)
proxy = filter_to_training_genes(proxy, training_genes)

#### Selection of high-confidence damaging variants (PS3)
We assemble a per-gene set of high-confidence damaging missense variants (PS3) from text-mining, MAVE, and curated functional datasets, prioritizing sources with stronger functional evidence and removing overlaps across sources.

In [3]:
import pandas as pd

alpha = 0.05 / 274 # Bonferroni correction for 274 MAVE studies
TARGET_PER_GENE = 25 # Target damaging variants per gene

df_out = variant_set.copy()

def get_concordant_ids(df, id_col="ID", label_col="Effect"):
    effect_counts = df.groupby(id_col)[label_col].nunique()
    valid_ids = effect_counts[effect_counts == 1].index
    return df[df[id_col].isin(valid_ids)].drop_duplicates()

def filter_ps3(df):
    return df[df["Effect"] == "PS3"].copy()

def drop_overlaps_in_order(dfs, key="ID"):
    used_ids = set()
    cleaned = {}
    for name, d in dfs:
        if used_ids:
            d = d[~d[key].isin(used_ids)].copy()
        cleaned[name] = d
        used_ids.update(d[key])
    return cleaned

def sample_up_to_n(df, n, random_state=42):
    if len(df) <= n:
        return df
    return df.sample(n=n, random_state=random_state)

text_mining_cc = get_concordant_ids(text_mining_training)
mave_cc = get_concordant_ids(mave_training)
curation_cc = get_concordant_ids(curation_training)

mave_99 = mave_cc[(mave_cc["AUC"] >= 0.99)  & (mave_cc["p_value"] <= alpha)]
mave_95 = mave_cc[(mave_cc["AUC"] >= 0.947) & (mave_cc["p_value"] <= alpha)]
mave_90 = mave_cc[(mave_cc["AUC"] >= 0.90)  & (mave_cc["p_value"] <= alpha)]

text_mining_ps3 = filter_ps3(text_mining_cc)
mave_99_ps3 = filter_ps3(mave_99)
mave_95_ps3 = filter_ps3(mave_95)
mave_90_ps3 = filter_ps3(mave_90)
curation_ps3 = filter_ps3(curation_cc)

ordered_sources = [
    ("text_mining", text_mining_ps3),
    ("mave_99", mave_99_ps3),
    ("mave_95", mave_95_ps3),
    ("mave_90", mave_90_ps3),
    ("curation", curation_ps3),
]

cleaned_sources = drop_overlaps_in_order(ordered_sources, key="ID")

text_mining_ps3 = cleaned_sources["text_mining"]
mave_99_ps3 = cleaned_sources["mave_99"]
mave_95_ps3 = cleaned_sources["mave_95"]
mave_90_ps3 = cleaned_sources["mave_90"]
curation_ps3 = cleaned_sources["curation"]

all_genes = sorted(
    set(text_mining_ps3["ensg"])
    | set(mave_99_ps3["ensg"])
    | set(mave_95_ps3["ensg"])
    | set(mave_90_ps3["ensg"])
    | set(curation_ps3["ensg"])
)

sources_by_trust = [
    ("text_mining", text_mining_ps3),
    ("mave_99", mave_99_ps3),
    ("mave_95", mave_95_ps3),
    ("mave_90", mave_90_ps3),
    ("curation", curation_ps3),
]

selected_rows = []

for gene in all_genes:
    remaining = TARGET_PER_GENE

    for src_name, src_df in sources_by_trust:
        if remaining <= 0:
            break

        gene_df = src_df[src_df["ensg"] == gene]
        if gene_df.empty:
            continue

        n_take = min(remaining, len(gene_df))
        picked = sample_up_to_n(gene_df, n_take, random_state=42)

        selected_rows.append(picked[["ID", "ensg", "Effect"]])
        remaining -= n_take

if selected_rows:
    damaging_set = pd.concat(selected_rows, ignore_index=True).drop_duplicates()
else:
    damaging_set = pd.DataFrame(columns=["ID", "ensg", "Effect"])

damaging_set


,ID,ensg,Effect
0,20-50935173-A-G,ENSG00000000419,PS3
1,20-50942086-G-A,ENSG00000000419,PS3
2,20-50942070-C-A,ENSG00000000419,PS3
3,1-196747182-C-T,ENSG00000000971,PS3
4,1-196747189-C-T,ENSG00000000971,PS3
...,...,...,...
5163,15-71811590-C-T,ENSG00000278570,PS3
5164,15-71817605-G-C,ENSG00000278570,PS3
5165,15-71814024-T-C,ENSG00000278570,PS3
5166,15-71813566-C-G,ENSG00000278570,PS3


#### Selection of neutral (BS3) and proxy benign variants
For each gene in the damaging set, we select an equal number of neutral (BS3) variants from functional datasets and then top up with high-frequency proxy benign missense variants, while avoiding overlaps and enforcing per-gene balance.

In [4]:
DROP_EXCESS_PS3 = True
AF_CUTOFF = 1 / 10_000

# Exclude very rare population variants from proxy set
proxy = proxy[proxy["Average_AF"] >= AF_CUTOFF]

genes_train = sorted(damaging_set["ensg"].unique())
damaging_ids = set(damaging_set["ID"])

def filter_bs3(df):
    return df[df["Effect"] == "BS3"].copy()

bs3_sources = [
    ("text_mining", filter_bs3(text_mining_cc)),
    ("mave_99",     filter_bs3(mave_99)),
    ("mave_95",     filter_bs3(mave_95)),
    ("mave_90",     filter_bs3(mave_90)),
    ("curation",    filter_bs3(curation_cc)),
]

bs3_sources = drop_overlaps_in_order(bs3_sources, key="ID")

text_mining_bs3 = bs3_sources["text_mining"]
mave_99_bs3     = bs3_sources["mave_99"]
mave_95_bs3     = bs3_sources["mave_95"]
mave_90_bs3     = bs3_sources["mave_90"]
curation_bs3     = bs3_sources["curation"]

for name, df_src in [
    ("text_mining_bs3", text_mining_bs3),
    ("mave_99_bs3",     mave_99_bs3),
    ("mave_95_bs3",     mave_95_bs3),
    ("mave_90_bs3",     mave_90_bs3),
    ("curation_bs3",    curation_bs3),
]:
    df_filtered = df_src[~df_src["ID"].isin(damaging_ids)].copy()
    if name == "text_mining_bs3":
        text_mining_bs3 = df_filtered
    elif name == "mave_99_bs3":
        mave_99_bs3 = df_filtered
    elif name == "mave_95_bs3":
        mave_95_bs3 = df_filtered
    elif name == "mave_90_bs3":
        mave_90_bs3 = df_filtered
    elif name == "curation_bs3":
        curation_bs3 = df_filtered

# Per-gene BS3 selection to balance PS3
bs3_sources_by_trust = [
    ("text_mining_bs3", text_mining_bs3),
    ("mave_99_bs3",     mave_99_bs3),
    ("mave_95_bs3",     mave_95_bs3),
    ("mave_90_bs3",     mave_90_bs3),
    ("curation_bs3",    curation_bs3),
]

neutral_rows = []
used_negative_ids = set()

for gene in genes_train:
    n_pos = (damaging_set["ensg"] == gene).sum()
    remaining = n_pos

    for src_name, src_df in bs3_sources_by_trust:
        if remaining <= 0:
            break

        gene_df = src_df[
            (src_df["ensg"] == gene)
            & (~src_df["ID"].isin(used_negative_ids))
            & (~src_df["ID"].isin(damaging_ids))
        ]

        if gene_df.empty:
            continue

        n_take = min(remaining, len(gene_df))
        picked = sample_up_to_n(gene_df, n_take, random_state=42).copy()

        neutral_rows.append(picked[["ID", "ensg", "Effect"]])
        used_negative_ids.update(picked["ID"])
        remaining -= n_take

if neutral_rows:
    neutral_train_set = (
        pd.concat(neutral_rows, ignore_index=True)
        .drop_duplicates(subset=["ID"])
    )
else:
    neutral_train_set = pd.DataFrame(columns=["ID", "ensg", "Effect"])


def topup_with_proxy(
    neutral_train_set,
    damaging_set,
    proxy,
    genes_train,
    used_ids,
):
    required = {"ID", "ensg", "Average_AF"}
    missing = required - set(proxy.columns)
    if missing:
        raise ValueError(f"proxy is missing required columns: {missing}")

    damaging_ids = set(damaging_set["ID"])
    existing_negative_ids = (
        set(neutral_train_set["ID"]) if not neutral_train_set.empty else set()
    )

    proxy_candidates = proxy[
        (~proxy["ID"].isin(damaging_ids))
        & (~proxy["ID"].isin(used_ids))
        & (~proxy["ID"].isin(existing_negative_ids))
    ].copy()

    if proxy_candidates.empty:
        return neutral_train_set, used_ids

    neutral_counts = (
        neutral_train_set.groupby("ensg")["ID"].size()
        if not neutral_train_set.empty
        else pd.Series(dtype=int)
    )

    proxy_selected = []

    for gene in genes_train:
        n_pos = (damaging_set["ensg"] == gene).sum()
        if n_pos == 0:
            continue

        n_neg = int(neutral_counts.get(gene, 0))
        remaining = n_pos - n_neg
        if remaining <= 0:
            continue

        gene_proxy = proxy_candidates[proxy_candidates["ensg"] == gene]
        if gene_proxy.empty:
            continue

        gene_proxy = gene_proxy.sort_values("Average_AF", ascending=False)
        picked = gene_proxy.head(remaining).copy()
        if picked.empty:
            continue

        picked["Effect"] = "proxy_benign"

        proxy_selected.append(picked[["ID", "ensg", "Effect"]])
        used_ids.update(picked["ID"])
        proxy_candidates = proxy_candidates[~proxy_candidates["ID"].isin(picked["ID"])]

    if proxy_selected:
        proxy_set = pd.concat(proxy_selected, ignore_index=True)
        neutral_train_set = pd.concat([neutral_train_set, proxy_set], ignore_index=True)

    neutral_train_set = neutral_train_set.drop_duplicates(subset=["ID"])

    return neutral_train_set, used_ids


neutral_train_set, used_negative_ids = topup_with_proxy(
    neutral_train_set=neutral_train_set,
    damaging_set=damaging_set,
    proxy=proxy,
    genes_train=genes_train,
    used_ids=used_negative_ids,
)

# Optionally drop excess PS3 to match non-pathogenic variants per gene
if DROP_EXCESS_PS3 and not neutral_train_set.empty:
    neg_counts = neutral_train_set.groupby("ensg")["ID"].size()

    dmg = damaging_set.copy()
    indices_to_drop = []
    rng = np.random.RandomState(42)

    for gene, n_pos in dmg["ensg"].value_counts().items():
        n_neg = int(neg_counts.get(gene, 0))
        gene_idx = dmg.index[dmg["ensg"] == gene].to_numpy()

        if n_neg == 0:
            if gene_idx.size > 0:
                indices_to_drop.extend(gene_idx.tolist())
            continue

        if n_pos > n_neg and gene_idx.size > 0:
            n_drop = n_pos - n_neg
            drop_idx = rng.choice(gene_idx, size=n_drop, replace=False)
            indices_to_drop.extend(drop_idx.tolist())

    if indices_to_drop:
        damaging_set = dmg.drop(index=indices_to_drop).reset_index(drop=True)
        print(
            f"Dropped {len(indices_to_drop)} PS3 variants to balance per gene")

print("Damaging training set size:", damaging_set.shape)
print("Neutral training set size:", neutral_train_set.shape)

# Build functional training labels (categorical)
functional_training_labels = pd.concat(
    [
        damaging_set.assign(functional_training_label="PS3")[
            ["ID", "ensg", "functional_training_label"]
        ],
        neutral_train_set.assign(
            functional_training_label=lambda x: x["Effect"]
        )[["ID", "ensg", "functional_training_label"]],
    ],
    ignore_index=True,
).drop_duplicates(subset=["ID", "ensg"])

extra = functional_training_labels.merge(df_out[["ID", "ensg"]].drop_duplicates(), on=["ID", "ensg"], how="left", indicator=True).query('_merge == "left_only"')[["ID", "ensg"]].drop_duplicates()
df_out = pd.concat([df_out, extra], ignore_index=True)
df_out = df_out.merge(functional_training_labels[["ID", "ensg", "functional_training_label"]], on=["ID", "ensg"], how="left")

functional_training_labels


Dropped 802 PS3 variants to balance per gene
Damaging training set size: (4366, 3)
Neutral training set size: (4366, 3)


,ID,ensg,functional_training_label
0,20-50935173-A-G,ENSG00000000419,PS3
1,20-50942086-G-A,ENSG00000000419,PS3
2,20-50942070-C-A,ENSG00000000419,PS3
3,1-196747182-C-T,ENSG00000000971,PS3
4,1-196747189-C-T,ENSG00000000971,PS3
...,...,...,...
8727,15-71812093-T-C,ENSG00000278570,proxy_benign
8728,15-71812024-A-G,ENSG00000278570,proxy_benign
8729,15-71811966-G-A,ENSG00000278570,proxy_benign
8730,15-71813545-G-A,ENSG00000278570,proxy_benign


## Construct functional testing dataset (external evaluation set)

#### Remove training gene families (all homologs) from external evaluation set in order to fully avoid type 2 circularity

In [5]:
gene_families = pd.read_csv("../resources/gene_families/gene_families.txt", sep="\t")
eval_genes = gene_families[gene_families["External_Eval"] == 1]["ensg"].tolist()

def filter_to_eval_genes(df, eval_genes):
    return df[df["ensg"].isin(eval_genes)]

text_mining_eval = filter_to_eval_genes(text_mining, eval_genes)
mave_eval = filter_to_eval_genes(mave, eval_genes)
curation_eval = filter_to_eval_genes(curation, eval_genes)

#### Selection of high-confidence damaging variants (PS3)
Using the external evaluation gene families, we assemble a per-gene set of high-confidence damaging missense variants (PS3) from text-mining, MAVE, and curated functional datasets, prioritizing sources with stronger functional evidence and removing overlaps across sources.

In [6]:
alpha = 0.05 / 274
TARGET_PER_GENE = 50

def get_concordant_ids(df, id_col="ID", label_col="Effect"):
    effect_counts = df.groupby(id_col)[label_col].nunique()
    valid_ids = effect_counts[effect_counts == 1].index
    return df[df[id_col].isin(valid_ids)].drop_duplicates()

def filter_ps3(df):
    return df[df["Effect"] == "PS3"].copy()

def drop_overlaps_in_order(dfs, key="ID"):
    used_ids = set()
    cleaned = {}

    for name, d in dfs:
        if used_ids:
            d = d[~d[key].isin(used_ids)].copy()
        cleaned[name] = d
        used_ids.update(d[key])
    return cleaned

def sample_up_to_n(df, n, random_state=42):
    if len(df) <= n:
        return df
    return df.sample(n=n, random_state=random_state)

text_mining_cc = get_concordant_ids(text_mining_eval)
mave_cc = get_concordant_ids(mave_eval)
curation_cc = get_concordant_ids(curation_eval)

mave_99 = mave_cc[(mave_cc["AUC"] >= 0.99)  & (mave_cc["p_value"] <= alpha)]
mave_95 = mave_cc[(mave_cc["AUC"] >= 0.947) & (mave_cc["p_value"] <= alpha)]
mave_90 = mave_cc[(mave_cc["AUC"] >= 0.90)  & (mave_cc["p_value"] <= alpha)]

text_mining_ps3      = filter_ps3(text_mining_cc)
mave_99_ps3   = filter_ps3(mave_99)
mave_95_ps3   = filter_ps3(mave_95)
mave_90_ps3   = filter_ps3(mave_90)
curation_ps3  = filter_ps3(curation_cc)

ordered_sources = [
    ("text_mining",     text_mining_ps3),
    ("mave_99",  mave_99_ps3),
    ("mave_95",  mave_95_ps3),
    ("mave_90",  mave_90_ps3),
    ("curation", curation_ps3),
]

cleaned_sources = drop_overlaps_in_order(ordered_sources, key="ID")

text_mining_ps3      = cleaned_sources["text_mining"]
mave_99_ps3   = cleaned_sources["mave_99"]
mave_95_ps3   = cleaned_sources["mave_95"]
mave_90_ps3   = cleaned_sources["mave_90"]
curation_ps3  = cleaned_sources["curation"]

all_genes = sorted(
    set(text_mining_ps3["ensg"])
    | set(mave_99_ps3["ensg"])
    | set(mave_95_ps3["ensg"])
    | set(mave_90_ps3["ensg"])
    | set(curation_ps3["ensg"])
)

sources_by_trust = [
    ("text_mining",     text_mining_ps3),
    ("mave_99",  mave_99_ps3),
    ("mave_95",  mave_95_ps3),
    ("mave_90",  mave_90_ps3),
    ("curation", curation_ps3),
]

selected_rows = []

for gene in all_genes:
    remaining = TARGET_PER_GENE

    for src_name, src_df in sources_by_trust:
        if remaining <= 0:
            break

        gene_df = src_df[src_df["ensg"] == gene]
        if gene_df.empty:
            continue

        n_take = min(remaining, len(gene_df))
        picked = sample_up_to_n(gene_df, n_take, random_state=42)

        selected_rows.append(picked[["ID", "ensg", "Effect"]])
        remaining -= n_take

if selected_rows:
    damaging_set = pd.concat(selected_rows, ignore_index=True).drop_duplicates()
else:
    damaging_set = pd.DataFrame(columns=["ID", "ensg", "Effect"])

damaging_set


,ID,ensg,Effect
0,1-171636301-T-G,ENSG00000034971,PS3
1,1-171636302-C-G,ENSG00000034971,PS3
2,1-171636143-A-G,ENSG00000034971,PS3
3,1-171636338-G-A,ENSG00000034971,PS3
4,1-171636473-C-T,ENSG00000034971,PS3
...,...,...,...
607,9-95456391-G-A,ENSG00000185920,PS3
608,9-95461864-T-C,ENSG00000185920,PS3
609,9-95467344-T-G,ENSG00000185920,PS3
610,15-51341743-T-C,ENSG00000186417,PS3


#### Selection of neutral (BS3) variants
For each gene in the damaging set, we select an equal number of neutral (BS3) variants or try to balance as much as possible.

In [7]:
def filter_bs3(df):
    return df[df["Effect"] == "BS3"].copy()

bs3_sources = [
    ("text_mining_bs3", filter_bs3(text_mining_cc)),
    ("mave_99_bs3",     filter_bs3(mave_99)),
    ("mave_95_bs3",     filter_bs3(mave_95)),
    ("mave_90_bs3",     filter_bs3(mave_90)),
    ("curation_bs3",    filter_bs3(curation_cc)),
]

bs3_sources = drop_overlaps_in_order(bs3_sources, key="ID")

text_mining_bs3 = bs3_sources["text_mining_bs3"]
mave_99_bs3     = bs3_sources["mave_99_bs3"]
mave_95_bs3     = bs3_sources["mave_95_bs3"]
mave_90_bs3     = bs3_sources["mave_90_bs3"]
curation_bs3    = bs3_sources["curation_bs3"]

damaging_ids = set(damaging_set["ID"])

cleaned = {}
for name, df_src in [
    ("text_mining_bs3", text_mining_bs3),
    ("mave_99_bs3",     mave_99_bs3),
    ("mave_95_bs3",     mave_95_bs3),
    ("mave_90_bs3",     mave_90_bs3),
    ("curation_bs3",    curation_bs3),
]:
    cleaned[name] = df_src[~df_src["ID"].isin(damaging_ids)].copy()

text_mining_bs3 = cleaned["text_mining_bs3"]
mave_99_bs3     = cleaned["mave_99_bs3"]
mave_95_bs3     = cleaned["mave_95_bs3"]
mave_90_bs3     = cleaned["mave_90_bs3"]
curation_bs3    = cleaned["curation_bs3"]

genes_eval = sorted(damaging_set["ensg"].unique())

bs3_sources_by_trust = [
    ("text_mining_bs3", text_mining_bs3),
    ("mave_99_bs3",     mave_99_bs3),
    ("mave_95_bs3",     mave_95_bs3),
    ("mave_90_bs3",     mave_90_bs3),
    ("curation_bs3",    curation_bs3),
]

neutral_rows = []
used_neutral_ids = set()

for gene in genes_eval:
    n_pos = (damaging_set["ensg"] == gene).sum()
    remaining = n_pos

    for _, src_df in bs3_sources_by_trust:
        if remaining <= 0:
            break

        gene_df = src_df[
            (src_df["ensg"] == gene)
            & (~src_df["ID"].isin(used_neutral_ids))
        ]

        if gene_df.empty:
            continue

        n_take = min(remaining, len(gene_df))
        picked = sample_up_to_n(gene_df, n_take, random_state=42).copy()

        neutral_rows.append(picked[["ID", "ensg", "Effect"]])
        used_neutral_ids.update(picked["ID"])
        remaining -= n_take

if neutral_rows:
    neutral_eval_set = (
        pd.concat(neutral_rows, ignore_index=True)
        .drop_duplicates(subset=["ID"])
    )
else:
    neutral_eval_set = pd.DataFrame(columns=["ID", "ensg", "Effect"])

print("Damaging eval set size:", damaging_set.shape)
print("Neutral eval set size:", neutral_eval_set.shape)

functional_testing_labels = pd.concat(
    [
        damaging_set.assign(functional_testing_label="PS3")[
            ["ID", "ensg", "functional_testing_label"]
        ],
        neutral_eval_set.assign(functional_testing_label="BS3")[
            ["ID", "ensg", "functional_testing_label"]
        ],
    ],
    ignore_index=True,
).drop_duplicates(subset=["ID", "ensg"])

df_out = df_out.merge(
    functional_testing_labels[["ID", "ensg", "functional_testing_label"]],
    on=["ID", "ensg"],
    how="left",
)

functional_testing_labels


Damaging eval set size: (612, 3)
Neutral eval set size: (359, 3)


,ID,ensg,functional_testing_label
0,1-171636301-T-G,ENSG00000034971,PS3
1,1-171636302-C-G,ENSG00000034971,PS3
2,1-171636143-A-G,ENSG00000034971,PS3
3,1-171636338-G-A,ENSG00000034971,PS3
4,1-171636473-C-T,ENSG00000034971,PS3
...,...,...,...
966,5-96425870-A-C,ENSG00000175426,BS3
967,5-96399005-C-G,ENSG00000175426,BS3
968,5-96393045-G-A,ENSG00000175426,BS3
969,8-132134369-G-A,ENSG00000184156,BS3


## Construct a balanced clinical training dataset

#### Remove external evaluation gene families (all homologs) from clinical training set as well to avoid type 2 circularity

In [8]:
clinvar_jan2025 = filter_to_training_genes(clinvar_jan2025, training_genes)

#### Selection of balanced, high-confidence clinical labels
From the January 2025 ClinVar, we first remove any variants already used in the functional external evaluation set. Then restrict to genes present in the functional training set, keep only high-review-status submissions (≥2 stars) with clear clinical labels (P or B), and for each gene sample up to 25 pathogenic (P) and 25 benign (B) variants to build a balanced clinical training set.

In [9]:
clinvar_jan2025 = clinvar_jan2025[~clinvar_jan2025["ID"].isin(functional_testing_labels["ID"])]
clinvar_jan2025 = clinvar_jan2025[clinvar_jan2025["ensg"].isin(functional_training_labels["ensg"])]

clinvar_jan2025 = clinvar_jan2025.dropna(subset=['sig'])
clinvar_jan2025.rename(columns={"sig": "clinical_training_label"}, inplace=True)
clinvar_jan2025["rev_stat"] = clinvar_jan2025["rev_stat"].astype(str).str.replace(r"\.0$", "", regex=True)
clinvar_jan2025 = clinvar_jan2025[clinvar_jan2025["rev_stat"].isin(["2", "3", "4"])]

def balance_clinvar(clinvar, max_per_class=25):
    balanced = []

    for gene, group in clinvar.groupby("ensg"):
        p_variants = group[group["clinical_training_label"] == "P"]
        b_variants = group[group["clinical_training_label"] == "B"]

        n_p = len(p_variants)
        n_b = len(b_variants)

        if n_p == 0 or n_b == 0:
            continue  # Skip if one class is missing

        minority_count = min(n_p, n_b, max_per_class)

        p_sampled = p_variants.sample(min(minority_count, n_p), random_state=42)
        b_sampled = b_variants.sample(min(minority_count, n_b), random_state=42)

        balanced.append(p_sampled)
        balanced.append(b_sampled)

    return pd.concat(balanced, ignore_index=True)

clinical_training_set = balance_clinvar(clinvar_jan2025)

df_out = df_out.merge(clinical_training_set[["ID", "ensg", "clinical_training_label"]], on=["ID", "ensg"], how="left")

clinical_training_set

,ID,rev_stat,clinical_training_label,Gene,ensg
0,1-196745862-A-G,2,P,CFH,ENSG00000000971
1,1-196747207-T-C,2,P,CFH,ENSG00000000971
2,1-196673076-C-T,2,P,CFH,ENSG00000000971
3,1-196747189-C-T,2,P,CFH,ENSG00000000971
4,1-196743496-G-C,2,B,CFH,ENSG00000000971
...,...,...,...,...,...
7601,15-71812458-G-A,2,B,NR2E3,ENSG00000278570
7602,22-50523994-C-T,2,P,SCO2,ENSG00000284194
7603,22-50523835-C-T,2,P,SCO2,ENSG00000284194
7604,22-50524353-C-G,2,B,SCO2,ENSG00000284194


## Construct a clinical testing dataset (recent ClinVar variants)

We use the most recent ClinVar missense submissions as an external clinical testing set: after removing any variants already used in the functional training set, functional testing set, or clinical training set to avoid circularity, we retain variants with clear P/B labels and review status ≥1 star and, for each gene, randomly sample up to 100 pathogenic and 100 benign variants, balancing to the minority class to enforce absolute per-gene class balance.

In [10]:
clinvar = clinvar[~clinvar["ID"].isin(functional_training_labels["ID"])]
clinvar = clinvar[~clinvar["ID"].isin(functional_testing_labels["ID"])]
clinvar = clinvar[~clinvar["ID"].isin(clinical_training_set["ID"])]

clinvar = clinvar.dropna(subset=['sig'])
clinvar.rename(columns={"sig": "clinical_testing_label"}, inplace=True)
clinvar["rev_stat"] = clinvar["rev_stat"].astype(str).str.replace(r"\.0$", "", regex=True)
clinvar = clinvar[clinvar["rev_stat"].isin(["1", "2", "3", "4"])]

def balance_clinvar(clinvar, max_per_class=100):
    balanced = []

    for gene, group in clinvar.groupby("ensg"):
        p_variants = group[group["clinical_testing_label"] == "P"]
        b_variants = group[group["clinical_testing_label"] == "B"]

        n_p = len(p_variants)
        n_b = len(b_variants)

        if n_p == 0 or n_b == 0:
            continue  # Skip if one class is missing

        minority_count = min(n_p, n_b, max_per_class)

        p_sampled = p_variants.sample(min(minority_count, n_p), random_state=42)
        b_sampled = b_variants.sample(min(minority_count, n_b), random_state=42)

        balanced.append(p_sampled)
        balanced.append(b_sampled)

    return pd.concat(balanced, ignore_index=True)

clinical_testing_set = balance_clinvar(clinvar, max_per_class=100)

extra = clinical_testing_set.merge(df_out[["ID", "ensg"]].drop_duplicates(), on=["ID", "ensg"], how="left", indicator=True).query('_merge == "left_only"')[["ID", "ensg"]].drop_duplicates()
df_out = pd.concat([df_out, extra], ignore_index=True)
df_out = df_out.merge(clinical_testing_set[["ID", "ensg", "clinical_testing_label"]], on=["ID", "ensg"], how="left")

clinical_testing_set

,ID,rev_stat,clinical_testing_label,ensg,Gene
0,1-196747163-G-T,1,P,ENSG00000000971,CFH
1,1-196747209-G-A,1,P,ENSG00000000971,CFH
2,1-196673076-C-A,1,P,ENSG00000000971,CFH
3,1-196747212-T-C,1,P,ENSG00000000971,CFH
4,1-196673908-C-T,1,B,ENSG00000000971,CFH
...,...,...,...,...,...
2991,12-13563555-G-A,1,B,ENSG00000273079,GRIN2B
2992,12-13563187-T-C,1,B,ENSG00000273079,GRIN2B
2993,12-13564540-G-A,1,B,ENSG00000273079,GRIN2B
2994,12-13616570-G-A,1,B,ENSG00000273079,GRIN2B


## Export final labels for the training and testing datasets

In [11]:
df_out[["functional_training_label", "clinical_training_label","functional_testing_label", "clinical_testing_label"]] = df_out[["functional_training_label", "clinical_training_label","functional_testing_label", "clinical_testing_label"]].replace(r'^\s*$', np.nan, regex=True)
df_out = df_out.dropna(subset=["functional_training_label", "clinical_training_label","functional_testing_label", "clinical_testing_label"], how='all')
df_out.to_csv('../data/intermediate/variant_labels.txt', sep='\t', index=False)
df_out

,ID,ensg,functional_training_label,functional_testing_label,clinical_training_label,clinical_testing_label
92,10-100989312-G-A,ENSG00000107815,proxy_benign,NaN,B,NaN
96,10-100989331-G-A,ENSG00000107815,PS3,NaN,NaN,NaN
110,10-100989833-T-G,ENSG00000107815,NaN,NaN,P,NaN
160,10-101771551-G-A,ENSG00000107831,PS3,NaN,NaN,NaN
203,10-102396271-G-A,ENSG00000077150,proxy_benign,NaN,NaN,NaN
...,...,...,...,...,...,...
1110645,6-161785839-A-T,ENSG00000185345,PS3,NaN,NaN,NaN
1117941,2-108910527-C-T,ENSG00000135960,proxy_benign,NaN,NaN,NaN
1117942,1-46194359-C-G,ENSG00000117472,NaN,NaN,NaN,P
1117943,2-108929216-C-T,ENSG00000153201,NaN,NaN,NaN,P
